In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

players = spark.table("lh_silver_game.players_clean")
purchases = spark.table("lh_silver_game.purchases_clean")
ad_events = spark.table("lh_silver_game.ad_events_clean")

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 3, Finished, Available, Finished, False)

In [2]:
purchase_by_player = (
    purchases
    .groupBy("player_id")
    .agg(
        F.sum("price_usd").alias("purchase_revenue"),
        F.count("*").alias("purchase_count")
    )
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 4, Finished, Available, Finished, False)

In [3]:
ad_by_player = (
    ad_events
    .groupBy("player_id")
    .agg(
        F.sum("revenue_usd").alias("ad_revenue")
    )
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 5, Finished, Available, Finished, False)

In [4]:
player_monetization = (
    players
    .select(
        "player_id",
        "country",
        "platform",
        "acquisition_channel"
    )
    .join(
        purchase_by_player,
        on="player_id",
        how="left"
    )
    .join(
        ad_by_player,
        on="player_id",
        how="left"
    )
    .fillna({
        "purchase_revenue": 0.0,
        "purchase_count": 0,
        "ad_revenue": 0.0
    })
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 6, Finished, Available, Finished, False)

In [5]:
player_monetization = (
    player_monetization
    .withColumn(
        "is_payer",
        F.when(F.col("purchase_count") > 0, 1).otherwise(0)
    )
    .withColumn(
        "total_revenue",
        F.col("purchase_revenue") + F.col("ad_revenue")
    )
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 7, Finished, Available, Finished, False)

In [6]:
print("Player count:", player_monetization.count())

display(
    player_monetization.limit(10)
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 8, Finished, Available, Finished, False)

Player count: 50000


SynapseWidget(Synapse.DataFrame, 88c21de6-b238-4fc5-b4be-2d3a7b464577)

In [7]:
monetization_metrics = (
    player_monetization
    .groupBy("acquisition_channel")
    .agg(
        F.count("*").alias("total_players"),
        F.sum("is_payer").alias("payers"),
        F.sum("purchase_revenue").alias("purchase_revenue"),
        F.sum("ad_revenue").alias("ad_revenue"),
        F.sum("total_revenue").alias("total_revenue")
    )
    .withColumn(
        "payer_conversion",
        F.col("payers") / F.col("total_players")
    )
    .withColumn(
        "arpu",
        F.col("total_revenue") / F.col("total_players")
    )
    .withColumn(
        "arppu",
        F.when(
            F.col("payers") > 0,
            F.col("purchase_revenue") / F.col("payers")
        ).otherwise(0)
    )
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 9, Finished, Available, Finished, False)

In [8]:
display(
    monetization_metrics
    .orderBy("acquisition_channel")
)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 476c4156-5c5b-4357-bf1e-ad1e88d5f6da)

In [9]:
monetization_metrics.write.format("delta").mode("overwrite").saveAsTable("lh_gold_game.monetization_metrics")

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 11, Finished, Available, Finished, False)

In [10]:
df_check = spark.table("lh_gold_game.monetization_metrics")

print("Saved row count:", df_check.count())
display(df_check)

StatementMeta(, c0d6d6f5-0c87-4086-b388-c935adc55aca, 12, Finished, Available, Finished, False)

Saved row count: 6


SynapseWidget(Synapse.DataFrame, d98e99b5-0f35-4ba1-8db2-b6e19f5462e4)